<a href="https://colab.research.google.com/github/lricci03/Hands-on-ML/blob/main/c10_in_place_operators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Examples of allowed and non allowed in-place operations with PyTorch backpropagation

## Python in-place operations with int and lists

+= is in-place operator, but Int are immutable objects

In [ ]:
x = 5
y = x
y += 1
print(x)
print(y)

5
6


Lists are mutable objects, .append() is in-place operator

In [ ]:
l = [1,2,3]
m = l
m.append(4)
print('m = ', m, 'l = ', l)

m =  [1, 2, 3, 4] l =  [1, 2, 3, 4]


## PyTorch tensors

In [ ]:
import torch

In [ ]:
x = torch.tensor(5., requires_grad=True)
y = x
y += 2 # RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.

RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.

The following is allowed:
1. x.cos() creates a new tensor t=cos(x) and saves x for the derivative, it doesn't need t for backward propagation.
2. cos_() saves the input t and then modifies t in place -> cos(t)

In [ ]:
x = torch.tensor(5., requires_grad=True)
f = x.cos().cos_()
f.backward()

The following is NOT allowed:
1. x.exp() creates a new tensor t=exp(x)
2. cos_() saves the input t for its derivative and then modifies t in place -> cos(t)
3. When doing backward propagation we need t = exp(x) for the derivative of exp(), but t has been modified to cos(t), this raises an error

In [ ]:
x = torch.tensor(5., requires_grad=True)
f = x.exp().cos_()
f.backward()

RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor []], which is output 0 of ExpBackward0, is at version 1; expected version 0 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True, check_nan=False).

The following is allowed:
  1. create new tensor t=cos(x),
  2. exp_() computes exp(t) and writes it in place of t.
  During backward propagation PyTorch only needs exp(t) and x (to compute the derivative of cos(x)).

In [ ]:
x = torch.tensor(5., requires_grad=True)
f = x.cos().exp_()
f.backward()

The following code gives a RuntimeError

In [ ]:
z = torch.tensor(2., requires_grad=True)
v = z+1
f = v.cos()*v.sin_()
f.backward()
# v points to 3
# v.cos():
#  -create new space in memory containing cos(3) with a new pointer v2
#  -save that derivative is -sin(v)
#  -v still points to 3
# v.sin_():
#  -create new space in memory containing 3 with a new pointer v3
#  -save that the derivative is cos(v3)
#  -v now points to sin(3) !! This will give Error in backward propagation bc -sin(v) uses v !

RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor []], which is output 0 of SinBackward0, is at version 1; expected version 0 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True, check_nan=False).

The following **doesn't** give RuntimeError.

In [ ]:

z = torch.tensor(2., requires_grad=True)
v = z+1
f = v.cos_()*v.sin_()
# v points to 3
# v.cos_():
#  -allocate a new space in memory containg 3 with a new pointer v2
#  -save in memory the derivative -sin(v2)
#  -v now points to cos(3)
# v.sin_():
#  -allocate a new space in memory containing cos(3) with a new pointer v3
#  -save in memory the derivative cos(v3)
#  -v now points to sin(cos(3))

However the output is

In [ ]:
z = torch.tensor(2., requires_grad=True)
v = z+1
a = v.cos_() # a points at the same location as v which now contains cos(3)
b = v.sin_() # we modify the value in the location of v so now a is also sin(cos(3))
print(a,b)
w = z+1
f = w.cos_()*w.sin_() # w -> cos(3), w -> sin(cos(3)). Then multiply w * w = sin(cos(3))^2
f == a * b

tensor(-0.8360, grad_fn=<SinBackward0>) tensor(-0.8360, grad_fn=<SinBackward0>)


tensor(True)